In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# —— 1. 读数据 & 新增几个简单特征 ——  
# —— 1. Load data & add simple features ——  
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test  = pd.read_csv('/kaggle/input/titanic/test.csv')

def add_features(df):
    # a) 从 Name 提取称谓 Title  
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
    # b) 家庭规模 FamilySize  
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    # c) 船舱首字母 CabinLetter (缺失填 U)  
    df['CabinLetter'] = df['Cabin'].fillna('U').str[0]
    # d) 票价分箱 FareBin（四分位）  
    df['FareBin'] = pd.qcut(df['Fare'].fillna(-1), 4, labels=False)
    return df

train = add_features(train)
test  = add_features(test)

# —— 2. 定义特征列 ——  
num_cols = ['Age','Fare','FamilySize','SibSp','Parch','FareBin']
cat_cols = ['Pclass','Sex','Embarked','Title','CabinLetter']

# —— 3. Preprocessing Pipeline ——  
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')), 
    ('scale',  StandardScaler())
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore'))
])
preproc = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

# —— 4. RandomForest + RandomizedSearchCV ——  
rf = RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True, class_weight='balanced')

param_dist = {
    'clf__n_estimators': np.arange(100,601,50),
    'clf__max_depth':    [None] + list(np.arange(5,21,5)),
    'clf__min_samples_split': np.arange(2,11),
    'clf__min_samples_leaf':  np.arange(1,5),
    'clf__max_features': ['auto','sqrt','log2']
}

pipeline = Pipeline([
    ('preproc', preproc),
    ('clf',     rf)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist, 
    n_iter=20, cv=cv, scoring='accuracy', 
    random_state=42, n_jobs=-1, verbose=1
)

X = train.drop(columns=['Survived','PassengerId','Name','Ticket','Cabin'])
y = train['Survived']
search.fit(X, y)

print("最佳 CV 分数／Best CV score:", search.best_score_)

# —— 5. 在全量数据上训练 & 生成提交文件 ——  
best_model = search.best_estimator_
best_model.fit(X, y)

X_test = test.drop(columns=['PassengerId','Name','Ticket','Cabin'])
preds  = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    preds
})
submission.to_csv('submission.csv', index=False)
print("已生成 submission.csv / Submission created")


Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomFor

最佳 CV 分数／Best CV score: 0.8394827694432239
已生成 submission.csv / Submission created
